# 11 — Disk I/O Deep Dive

**Purpose:** isolate the **runtime startup disk overhead** of each language —
how much each runtime/binary reads from disk just to start, independent of the
workload. Feeds the thesis discussion of interpreter/runtime weight.

Method: benchmarks with **no input file** (binary-trees, fannkuch-redux, fasta,
mandelbrot, n-body, spectral-norm) generate their data in memory, so their disk
reads reflect only binary/runtime loading. Averaging reads over those benchmarks
gives the per-language startup overhead.

> The earlier exploratory disk views (per-benchmark heatmaps, read/write grids,
> strip plots, reads-vs-time scatter) were retired in the Phase 0 cut-list — disk
> I/O is otherwise negligible and not central to the thesis.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plot_style as ps
ps.apply_style()


def find_col(df, keyword):
    """Return the first column whose name contains `keyword`.

    Input: a DataFrame and a substring. Returns the matching column name;
    raises KeyError if none match (surfaces a renamed/missing column early)."""
    matches = [c for c in df.columns if keyword in c]
    if not matches:
        raise KeyError(f'No column matching "{keyword}"')
    return matches[0]

## 1. Load data

In [ ]:
runs    = ps.load_runs()
means   = ps.cell_means(runs)
COL_R   = find_col(runs, 'disk_total_read')

print(f'Per-run rows : {len(runs)}')
print(f'Cell-means   : {len(means)}')
print(f'Read column  : {COL_R}')

## 2. Runtime Startup Disk Overhead per Language

Mean disk reads on the no-input benchmarks — pure binary/interpreter load cost.
Bars are coloured by execution paradigm (Okabe–Ito palette).

In [ ]:
# No-input benchmarks: disk reads here = runtime/binary startup only.
NO_INPUT = ['binary-trees', 'fannkuch-redux', 'fasta',
            'mandelbrot', 'n-body', 'spectral-norm']

startup = (means[means['benchmark'].isin(NO_INPUT)]
           .groupby('language')[COL_R]
           .mean()
           .sort_values())

colors = [ps.paradigm_color(l) for l in startup.index]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(startup.index, startup.values, color=colors, edgecolor='white')

x_max = startup.max()
for bar, val in zip(bars, startup.values):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f} MB', va='center', ha='left', fontsize=8, color='#333333')

ax.set_xlim(0, x_max * 1.22)
ax.set_xlabel('Mean Disk Read (MB)')
ax.set_ylabel('Language')
ax.set_title('Runtime Startup Disk Overhead per Language\n'
             '(mean reads on no-input benchmarks — binary/interpreter load only)')
ax.grid(False)
ax.grid(axis='x', linestyle='--', alpha=0.5)
ax.legend(handles=ps.paradigm_handles(), title='Paradigm', loc='lower right')

plt.tight_layout()
ps.save_fig(fig, '11_runtime_startup_overhead')
plt.show()

> **Takeaway:** interpreted/VM runtimes carry the heaviest startup read cost
> (notably Ruby), while AOT-compiled native binaries load with the least disk I/O —
> a fixed overhead added on top of any actual input the workload reads.